# Week 1 Day 5 Lab 2 - Harness Engineering

## Deep Agents works the queue on its own

Deep Agents is LangChain's ready-made harness. It's built on `create_agent`, the same function we used on Day 2, with middleware added for file tools, subagents, a to-do list and summarizing a long conversation. It runs on any model LangChain supports, so we can use OpenRouter.

This is the same job as Lab 1: a folder with five claims from the queue, the policy wording and the claims manual, and the instruction to decide every claim. We don't write any tools.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls DeepSeek V4.1 Flash through OpenRouter, and a run on five claims costs about 3 cents.
            If you'd rather not spend anything, there's a free version of this lab in <a href="week1day5_lab2free.ipynb">week1day5_lab2free.ipynb</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_openai import ChatOpenAI

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Part 1: The claims desk

The claims desk is the same as in Lab 1. We use the five claims marked for the tiny version of the queue, which include one that needs the claims manual.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [4]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [5]:
DESK = Path("lab2_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

## Part 2: Setting up the harness

The instructions are the same as in Lab 1, apart from calling the task list a to-do list, which is Deep Agents' name for it.

In [6]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Deep Agents</h2>
            <span style="color:#118AB2;"><code>create_deep_agent</code> gives the agent file tools (<code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>) and <code>task</code>, which hands work to a subagent.<br/><br/>
            The to-do list is an extra since version 0.7, so we add it with LangChain's <code>TodoListMiddleware</code>, which gives the agent a <code>write_todos</code> tool.<br/><br/>
            <code>FilesystemBackend</code> points the file tools at the claims desk. With <code>virtual_mode=True</code> the agent sees the desk as <code>/</code> and can't reach anything outside it.
            </span>
        </td>
    </tr>
</table>

In [7]:
llm = ChatOpenAI(
    model="deepseek/deepseek-v4.1-flash",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort="low",
    timeout=120,
    max_retries=5,
)

In [8]:
claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware()],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)

## Part 3: Working the queue

As the agent works we print each step and count the model calls and the cost. The stream gives us an update after each step of the agent, and the updates from the `model` step hold the model's replies. The subagent's steps are indented.

In [9]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call["name"]

In [10]:
cost = 0
calls = 0

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            cost += message.response_metadata["token_usage"]["cost"]
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

print(f"\nFinished after {calls} model calls for ${cost:.4f}")

ls .
ls claims
I'll start by exploring the folder structure to understand what we're working with.
read_file /claims_manual.md
read_file /policy_wording.md
read_file /company.json
read_file /claims_manual.md
read_file /policy_wording.md
read_file /claims/CLM-1007.json
read_file /claims/CLM-1015.json
read_file /claims/CLM-1017.json
read_file /claims/CLM-1020.json
read_file /claims/CLM-1021.json
ls /policies
ls /customers
ls /decisions
Now let me read the claim files and the relevant policy/customer records.
read_file /policies/POL-2007.json
read_file /policies/POL-2015.json
read_file /policies/POL-2017.json
read_file /policies/POL-2020.json
read_file /policies/POL-2021.json
read_file /customers/CUS-3007.json
read_file /customers/CUS-3015.json
read_file /customers/CUS-3017.json
read_file /customers/CUS-3020.json
read_file /customers/CUS-3021.json
Let me read the relevant policies and customers.
write_todos: CLM-1007 (Kowalski, Economy, p (in_progress), CLM-1015 (Callister, First, tr (in_

## Part 4: How it did

We score the decision files with the same test as in Lab 1. A null payout counts as incorrect: the instructions require a numeric payout, with `0` for declined or referred claims.

In [12]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    payout = answer.get("payout", 0)
    if payout is None:
        return False
    return answer.get("decision") == label["decision"] and abs(float(payout) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

4 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50.0,approve,50.0,True
1,CLM-1015,decline,0.0,decline,0.0,True
2,CLM-1017,refer,NaN,refer,0.0,False
3,CLM-1020,approve,240.0,approve,240.0,True
4,CLM-1021,approve,825.0,approve,825.0,True


And here's the note it left for the handler.

In [13]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Handover — claims queue worked

All five claims in `claims/` have been decided; decision files are in `decisions/`.
Wording version 2026.1 / manual version 2026.2. No payout exceeds the $1,000 payment
authority limit, so none of these decisions needs a sign-off.

| Claim | Customer | Plan / add-ons | Decision | Payout |
|---|---|---|---|---|
| CLM-1007 | Greg Kowalski | Economy | Approve | $50.00 |
| CLM-1015 | Kate Callister | First | Decline | $0.00 |
| CLM-1017 | Emily Nguyen | Economy | Refer (documents) | — (indicative $420.00) |
| CLM-1020 | Nadia Rahman | First | Approve | $240.00 |
| CLM-1021 | Camille Fontaine | Premium + Electronics Plus | Approve | $825.00 |

## What was decided, and why

**CLM-1007 — Approve, $50.00.** Phone dropped in the hotel shower (accidental damage,
inside the trip 10–17 Jul, filed in time). Repair report + replacement receipt on file.
Economy per-item limit $150 caps the $1,099 claim; less the $100 Economy deductible = $50.
Wording s.5, s.1, s.10.

**CLM-1015 — Decline, $0.00.** Trip interruption: the family accidentally left an
8-year-old at home and the mother flew straight back from Paris. Sympathetic, but the
reason is not one of the five covered reasons in wording s.2, which ends "We do not pay
for cancellation or interruption for any other reason." Documents and timing were fine —
the claim fails on the covered reason itself. Wording s.2.

**CLM-1017 — Refer, document request.** Backpack snatched by a scooter rider in Naples
(theft from the person, so not unattended — covered in principle). Two required documents
are missing: the Carabinieri police report (customer says it exists but hasn't sent it) and
proof of value for the $130 sunglasses. Manual s.5 requires a refer with a named document
request, not a decline and not an approval on a promise. Indicative payout if the documents
come through: backpack $90 + tablet $150 (capped) + rain jacket $150 (capped) + sunglasses
$130 = $520, less the $100 deductible = **$420**. 30-day clock is running; claim stays open.

**CLM-1020 — Approve, $240.00.** Prescription sunglasses taken off the customer's face by a
macaque at Uluwatu Temple and dropped over the cliff. Under manual s.7 a wild-animal loss is
accidental loss, **not theft**, so no police report is required (and the police genuinely
don't take monkey reports) and the moths/vermin/insects/pets exclusion does not apply. The
licensed guide's statement is accepted evidence; optician receipt proves the $240. First
plan, no deductible, within the $500 per-item limit. Wording s.5, manual s.7.

**CLM-1021 — Approve, $825.00.** Tote bag snatched from the shoulder in Buenos Aires,
reported to police the same day. Covered theft. **Watch the Electronics Plus definition**
(wording s.8): it covers phones, tablets, laptops, cameras/lenses, e-readers and
smartwatches — nothing else. So the e-reader ($180) gets the raised $1,200 limit, but the
headphones ($429 → $300) and the handheld game console ($399 → $300) fall under the ordinary
$300 per-item limit. Tote $95 (under $100, so the customer's own list is proof of value).
$300 + $180 + $300 + $95 = $875, less the $50 Premium deductible = **$825**. Wording s.5, s.8, s.1.

## Things to know

- **Only referral in the batch is CLM-1017** — it needs the police report and sunglasses
  proof of value, then it can be approved at $420 (no sign-off needed). Nothing else is
  waiting on a person.
- **No SIU referrals.** Every customer here has zero prior claims, so the third-claim-in-12-
  months rule (manual s.9) never triggered. No story-vs-document conflicts either — no dates,
  receipts or carrier records contradicted the customer, so nothing went to Special Investigations.
- **No sign-offs needed.** The highest payout is $825, below the $1,000 limit in
  company.json / manual s.3.
- **Two claims turned on interpretation notes worth remembering:** wild animals are treated
  as accidental loss, not theft (CLM-1020); and the Electronics Plus limit is defined narrowly
  and applied item by item (CLM-1021). Both are places a quick read could go wrong.
- **One decline (CLM-1015)** rests purely on the closed list of covered reasons in s.2 — the
  story is unusual but that is not a reason to approve an excluded claim (manual s.2/s.7).


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">Deep Agents is open source and runs on any model, so you can pick a cheap one and read exactly what the harness does. The Claude Agent SDK gives you Claude Code's harness, tuned for Claude models.<br/><br/>
            Next we build the same pieces into UnderCover ourselves: a to-do list, tools to read the manual, and a subagent for each claim.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its to-do list and where it handed work to the subagent.<br/><br/>
            Then try other models and compare accuracy, cost and time: <code>openai/gpt-6-luna</code>, or <code>anthropic/claude-sonnet-5.5</code> to compare with Lab 1 on the same model, which costs more.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>